In [ ]:
import oracledb
import pandas as pd

# -------- CONFIG --------
START_DATE = "2025-04-01 00:00"
END_DATE   = "2025-11-26 23:59"

# Oracle Client init (safe even if called earlier)
try:
    oracledb.init_oracle_client(
        lib_dir=r"D:/app/Oracle11gX64/product/11.2.0/client_1/BIN"
    )
except oracledb.ProgrammingError:
    # Already initialized in this process
    pass

dsn = oracledb.makedsn("157.0.2.23", 1521, service_name="imdb")

# -------- COLUMN LIST (from your DAILY query) --------
COLUMNS = [
    "TIMESTAMP",
    "ZONE1_TEMP", "ZONE2_TEMP", "ZONE3_TEMP", "ZONE4_TEMP", "ZONE5_TEMP",
    "ZONE6_TEMP", "ZONE7_TEMP", "ZONE8_TEMP", "ZONE9_TEMP", "ZONE10_TEMP",
    "ZONE11_TEMP",
    "IRON_ORE_FEED_RATE", "FD_COAL_FEED_RATE", "DOLO_FEED_RATE",
    "FINES_COAL_FEED_RATE", "CC_FEED_RATE", "FEED_MIX_FEED_RATE", "KMD_RPM",
    "CHAR_FEED_RATE", "FINES_FEED_RATE", "LUMPS_FEED_RATE",
    "IRON_ORE_FEED_RATE_PLUS_STBY", "FEED_COAL_FR_MAIN_PLUS_STBY",
    "ELBOW_DUCT_TEMP", "INLET_HOOD_TEMP", "OUTLET_HOOD_TEMP", "STEAM_FLOW",
    "CD_TEMPERATURE", "WHRB_DAMPER_POSITION", "BOILER_INLET_TEMP",
    "WHRB_DUST_EMMISION",
    "UPPER_ABC_ZONE3_TEMP", "UPPER_ABC_ZONE4_TEMP", "UPPER_ABC_ZONE5_TEMP",
    "LOWER_ABC_ZONE6_TEMP", "LOWER_ABC_ZONE7_TEMP", "LOWER_ABC_ZONE8_TEMP",
    "DSC_BACKFLOW_TEMP", "DSC_ZONE10_TEMP", "DSC_ZONE11_TEMP",
    "LOBE_PRESSURE", "KILN_INLET_PRESSURE", "KILN_OUTLET_PRESSURE",
    "COOLER_INLET_PRESSURE", "COOLER_OUTLET_PRESSURE",
    "SAF_FAN_SET_POINT", "KMD_RPM_SET_POINT", "FINES_COAL_SET_POINT",
    "COARSE_COAL_SET_POINT", "IRON_ORE_SET_POINT_STBY", "IRON_ORE_SET_POINT",
    "FEED_COAL_SET_POINT_STBY", "FEED_COAL_SET_POINT", "DOLOMITE_SET_POINT",
    "CB_FAN_SET_POINT", "OVER_SIZE_SET", "FINES_SET", "LUMPS_SET",
    "KMD_CURRENT_MASTER", "KMD_CURRENT_SLAVE", "LOBE_FLOW", "CB_AIR_FLOW",
    "CB__CURRENT", "SAF_CURRENT", "CD_BELT_SPEED", "CD_HOPPER_LEVEL",
    "CD_MATERIAL_FLOW", "COOLER_WATER_FLOW",
    "U_ABC_AIR_FLOW", "M_ABC_AIR_FLOW",
    "U_ABC_CURRENT", "M_ABC_CURRENT", "L_ABC_CURRENT", "CMD_CURRENT",
    "KMD_MASTER_RTD1", "KMD_MASTER_RTD2", "KMD_SLAVE_RTD1",
    "KMD_SLAVE_RTD2",
    "WHRB_CURRENT",
    "IRON_ORE_BUNKER_LEVEL", "FD_COAL_BUNKER_LEVEL", "DOLO_BUNKER_LEVEL",
    "FINES_COAL_BUNKER_LEVEL", "CC_BUNKER_LEVEL",
    "WHRB_O2", "WHRB_CO2", "WHRB_CO_PPM", "WHRB_NO_PPM", "WHRB_NO2_PPM",
    "WHRB_NOX_PPM", "WHRB_SO2_PPM",
    "COMMON_BC_15_FLOW", "COMMON_DRI_FINES_FLOW", "COMMON_DRI_CHAR_FLOW",
    "COMMON_DRI_LUMP_FLOW",
    "SAF_1_SECONDARY_AIR", "SAF_2__SECONDARY_AIR", "SAF_3_SECONDARY_AIR",
    "SAF_4_SECONDARY_AIR", "SAF_5_SECONDARY_AIR", "SAF_6_SECONDARY_AIR",
    "SAF_7_SECONDARY_AIR", "SAF_8_SECONDARY_AIR",
    "COM_SILO_LEVEL", "COM_SILO_BAG_FILTER_DPT", "COM_SINGLE_DRUM_SPEED",
    "COM_DOUBLE_DRUM1_SPEED", "COM_DOUBLE_DRUM2_SPEED",
    "LUMPS_PRODUCTION_PD", "FINES_PRODUCTION_PD", "CHAR_PRODUCTION_PD",
    "IRON_ORE_CONSUMPTION_PD", "DOLOMITE_CONSUMPTION_PD",
    "FEED_COAL_CONSUMPTION_PD", "COARSE_COAL_CONSUMPTION_PD",
    "FINES_COAL_CONSUMPTION_PD", "COOLER_DISCHARGE_PD",
    "STEAM_PRODUCTION_PD", "PRODUCTION_FACTOR", "FINES_DAY_TOTAZIER",
    "PRODUTION_WET_SCRAPPER", "PRODUTION_LOW_MAG",
    "PRODUTION_COOLER_OVER_SIZE", "PRODUTION_CHAR_COAL",
    "LUMPS_DAY_TOTAZIER", "CHAR_DAY_TOTAZIER",
    "IRON_ORE_LOW_GRED_FACTOR", "IRON_ORE_HIGH_GRED_FACTOR",
    "ABC_GUN_CONTROL_VALVE1", "ABC_GUN_CONTROL_VALVE2",
    "ABC_GUN_CONTROL_VALVE3", "ABC_GUN_CONTROL_VALVE4",
    "ABC_GUN_CONTROL_VALVE5", "ABC_GUN_CONTROL_VALVE6",
    "ABC_GUN_CONTROL_VALVE7", "ABC_GUN_CONTROL_VALVE8",
    "ABC_GUN_CONTROL_VALVE9"
]

# Build SELECT list with table alias and quoted identifiers
columns_sql = ",\n    ".join([f't."{col}"' for col in COLUMNS])

# -------- SQL FOR DAILY DATA --------
sql_daily = f"""
SELECT
    {columns_sql}
FROM TSBSL.T_TSM_DRI_K1_PROCESS_DAILY t
WHERE t."TIMESTAMP" >= TO_TIMESTAMP(:1, 'YYYY-MM-DD HH24:MI')
  AND t."TIMESTAMP" <= TO_TIMESTAMP(:2, 'YYYY-MM-DD HH24:MI')
ORDER BY t."TIMESTAMP"
"""

# -------- EXECUTE AND BUILD FINAL DATASET --------
with oracledb.connect(user="imonitor", password="imtg", dsn=dsn) as conn:
    df_proc = pd.read_sql(sql_daily, con=conn, params=[START_DATE, END_DATE])

# Ensure TIMESTAMP is datetime and set as index
df_proc["TIMESTAMP"] = pd.to_datetime(df_proc["TIMESTAMP"])
df_proc = df_proc.set_index("TIMESTAMP").sort_index()

df_proc = df_proc.copy()
df_proc['SAMPLE_DATE'] = df_proc.index.date

# At this point df_daily is your final day-wise dataset
print("Daily dataframe shape:", df_proc.shape)
print(df_proc.head())


In [ ]:
import oracledb
import pandas as pd
import datetime as dt

# -------- 1. CONFIG: dates ---------------------------------------
start_ts = dt.datetime(2025, 4, 1, 0, 0, 0) # FROM date/time
end_ts = dt.datetime(2025, 11, 26, 23, 59, 59) # TO date/time

# -------- 2. Oracle client + connection --------------------------
# Use the same lib_dir, user, password, dsn that are already working for you

# -------- 3. SQL query with named bind variables -----------------
sql = """
SELECT
    SAMPLE_DATE,
    CASE
        WHEN GROUPING(DEPT) = 1 THEN 'DRI-ALL'
        ELSE DEPT
    END AS DEPT,
    'COOLER DISCHARGE MAGNETIC' AS TYPE,
    ROUND(AVG(FE_M_4_TO_22), 3) AS AVG_FE_M_4_22,
    COUNT(*) AS ROWS_
FROM (
    SELECT
        TRUNC(CAST(t."TIMESTAMP" AS DATE)) AS SAMPLE_DATE,
        t.DEPT,
        t.FE_M_4_TO_22
    FROM TSBSL.T_TSM_DRI_CHEM_ANALYSIS t
    WHERE t."TIMESTAMP" BETWEEN :start_ts AND :end_ts
      AND t.DEPT LIKE 'DRI-1'
      AND t.TYPE LIKE 'COOLER DISCHARGE MAG%'
)
GROUP BY ROLLUP (SAMPLE_DATE, DEPT)
HAVING GROUPING(SAMPLE_DATE) = 0
ORDER BY SAMPLE_DATE, DEPT
"""

# -------- 4. Run query into a pandas DataFrame -------------------
params = {"start_ts": start_ts, "end_ts": end_ts}

with oracledb.connect(user="iMonitor", password="imtg", dsn=dsn) as conn:
    df_quality = pd.read_sql(sql, conn, params=params)
    df_quality = df_quality[df_quality['DEPT']!= 'DRI-ALL'].copy()
    df_quality = df_quality[['SAMPLE_DATE', 'AVG_FE_M_4_22']].copy()
    df_quality['SAMPLE_DATE'] = pd.to_datetime(df_quality['SAMPLE_DATE'])
    df_quality['DATE_KEY'] = df_quality['SAMPLE_DATE'].dt.strftime('%Y-%m-%d')
    
    print(df_quality.head())
    print(df_quality.dtypes)
    
    print(df_quality.head(100))


# df_proc already built and TIMESTAMP is the index
# create SAMPLE_DATE from index
df_proc = df_proc.copy()
df_proc['SAMPLE_DATE'] = df_proc.index # this is datetime64[ns]
df_proc['DATE_KEY'] = df_proc['SAMPLE_DATE'].dt.strftime('%Y-%m-%d')

print(df_proc[['SAMPLE_DATE', 'DATE_KEY']].head())
print(df_proc.dtypes[['SAMPLE_DATE', 'DATE_KEY']])

In [ ]:

df_merged = pd.merge(
    df_proc.reset_index(), # bring TIMESTAMP back as a column
    df_quality[['DATE_KEY', 'AVG_FE_M_4_22']], # only one Fe% column + key
    on='DATE_KEY',
    how='inner'
)

print(df_merged.head())
print("Merged shape:", df_merged.shape)



In [ ]:


import seaborn as sns
import matplotlib.pyplot as plt

# Select only numeric columns
num_df = df_merged.select_dtypes(include='number')

# Compute correlation only with FeM
target_col = 'AVG_FE_M_4_22'
corr_series = num_df.corr()[target_col].sort_values(ascending=False)

# Convert to DataFrame for heatmap
corr_df = corr_series.to_frame().reset_index()
corr_df.columns = ['Parameter', 'Correlation']

# Plot Heatmap
plt.figure(figsize=(6, len(corr_df)*0.25 + 4))
sns.heatmap(corr_df.set_index('Parameter'),
            annot=True,
            cmap='coolwarm',
            center=0,
            linewidths=0.5,
            cbar=True)

plt.title('Correlation: Process Parameters vs AVG_FE_M_4_22', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Value')
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd

# --- 1. CONFIGURATION ---
# Target column (Quality)
target_col = 'AVG_FE_M_4_22'

# Assuming 'df_merged' is your complete DataFrame
# --- 2. DATA PREPARATION ---
# Select only numeric columns from the DataFrame
# This step is still important for filtering out text/category columns.
num_df = df_merged.select_dtypes(include='number')

# --- 3. CORRELATION CALCULATION AND SORTING ---
# KEY CHANGE: pandas corr() automatically excludes missing data (NaNs)
# in a pairwise fashion (meaning it only uses rows where BOTH variables are present).
# If a column is mostly or entirely NaN, the correlation against the target will be NaN.
# To ensure the correlation calculation runs cleanly:
# 1. We drop the target column itself for the series.
# 2. We sort the resulting series.
corr_series = num_df.corr(method='pearson')[target_col].drop(target_col)

# IMPORTANT FIX: Drop any resulting NaN correlations before sorting, 
# as these are caused by columns that are mostly or entirely NaN.
corr_series = corr_series.dropna() 

# Now sort the cleaned series from most positive (strongest positive) 
# to most negative (strongest negative) correlation.
corr_series_sorted = corr_series.sort_values(ascending=False)


# --- 4. SELECT TOP 10 ---
# The top 10 values in the sorted series are the 10 most positive correlations.
top10_pos = corr_series_sorted.head(10)

# The bottom 10 values in the sorted series are the 10 most negative correlations.
top10_neg = corr_series_sorted.tail(10)

# --- 5. PRINT RESULTS ---
print(f"===== Top 10 POSITIVE Correlations vs {target_col} =====")
print(top10_pos)

print(f"\n===== Top 10 NEGATIVE Correlations vs {target_col} =====")
print(top10_neg)

# --- 6. VISUALIZATION: Heatmap for Top 10 Positive ---
plt.figure(figsize=(7, 6))
sns.heatmap(top10_pos.to_frame(),
            annot=True,
            cmap='coolwarm', 
            center=0,
            linewidths=0.6,
            cbar=True,
            fmt=".3f") 
plt.title(f'Top 10 POSITIVE correlations vs {target_col}',
          fontsize=13, fontweight='bold')
plt.xlabel('Correlation Value')
plt.tight_layout()
plt.show()

# --- 7. VISUALIZATION: Heatmap for Top 10 Negative ---
plt.figure(figsize=(7, 6))
sns.heatmap(top10_neg.to_frame(),
            annot=True,
            cmap='coolwarm', 
            center=0,
            linewidths=0.6,
            cbar=True,
            fmt=".3f") 
plt.title(f'Top 10 NEGATIVE correlations vs {target_col}',
          fontsize=13, fontweight='bold')
plt.xlabel('Correlation Value')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- 1. CONFIGURATION ---
TARGET_COL = 'AVG_FE_M_4_22'

# Parameters to EXCLUDE from plotting as requested
EXCLUSIONS = [
    'ABC_GUN_CONTROL_VALVE2', 'ABC_GUN_CONTROL_VALVE1', 'ABC_GUN_CONTROL_VALVE8',
    'ABC_GUN_CONTROL_VALVE5', 'ABC_GUN_CONTROL_VALVE6', 'ABC_GUN_CONTROL_VALVE7', 
    'ABC_GUN_CONTROL_VALVE9', 'LOWER_ABC_ZONE6_TEMP', 'LOWER_ABC_ZONE7_TEMP',
    'LOWER_ABC_ZONE8_TEMP', 'ZONE2_TEMP'
]

# --- 2. COMPILE ALL RELEVANT PARAMETERS ---

# Get the index (parameter names) from your Top 10 lists
all_top_parameters = list(top10_pos.index) + list(top10_neg.index)

# Remove duplicates if any parameter appears in both lists (unlikely but safe)
unique_parameters = set(all_top_parameters)

# --- 3. FILTER PARAMETERS FOR PLOTTING ---

# Filter out the excluded parameters
plot_parameters = [
    param for param in unique_parameters if param not in EXCLUSIONS
]

print("--- Parameters Selected for Scatter Plotting ---")
print(plot_parameters)
print("-" * 50)

# --- 4. GENERATE SCATTER PLOTS ---

# Calculate the number of plots needed
num_plots = len(plot_parameters)
# Determine the layout (e.g., 3 columns)
cols = 3
rows = int(np.ceil(num_plots / cols))

plt.figure(figsize=(5 * cols, 4 * rows))
plt.suptitle(f'Correlation Plots: {TARGET_COL} vs. Key Process Parameters', fontsize=16, y=1.02)

for i, param in enumerate(plot_parameters):
    plt.subplot(rows, cols, i + 1)
    
    # Calculate the correlation for the title
    corr_value = df_merged[[TARGET_COL, param]].corr().iloc[0, 1]
    
    # Create the scatter plot
    # Use dropna(subset=[param, TARGET_COL]) to ensure the plot is clean
    sns.scatterplot(
        data=df_merged.dropna(subset=[param, TARGET_COL]), 
        x=param, 
        y=TARGET_COL,
        alpha=0.6,
        color='darkblue'
    )
    
    # Add a title with the correlation strength
    plt.title(f'{param} (Corr: {corr_value:.3f})', fontsize=10)
    plt.xlabel(param, fontsize=9)
    plt.ylabel(TARGET_COL, fontsize=9)
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()